In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from feature_engineering import build_feature_dataframe

In [2]:
# ── 1. Load & clean ───────────────────────────────────────────────────────────
data = pd.read_csv("data/vcbench_final_public.csv").replace({np.nan: None})
data.head()

,founder_uuid,success,industry,ipos,acquisitions,educations_json,jobs_json,anonymised_prose
0,33159ebb-97ff-43fe-a80e-31fdcf467065,1,"Technology, Information & Internet Platforms",None,None,"[\n {\n ""degree"": ""BA"",\n ""field"": ""Com...","[\n {\n ""role"": ""CTO"",\n ""company_size""...",This founder leads a startup in the Technology...
1,33a7bba0-2ef6-415b-b73c-3dc994b8a86e,1,Entertainment & Live Arts,None,None,"[\n {\n ""degree"": """",\n ""field"": """",\n ...","[\n {\n ""role"": ""Board of Directors, VP"",\...",This founder leads a startup in the Entertainm...
2,0fe9fcdf-eb06-4e2c-88d8-04468b427298,1,Industrial & Agricultural Machinery Manufacturing,None,None,None,"[\n {\n ""role"": ""Security Expert"",\n ""c...",This founder leads a startup in the Industrial...
3,4f5620d4-9db8-4cfc-a1f5-2fd917472865,1,Research Services & Market Analysis,None,None,"[\n {\n ""degree"": ""PhD"",\n ""field"": ""Ph...","[\n {\n ""role"": ""Professor"",\n ""company...",This founder leads a startup in the Research S...
4,c347a753-2280-48f8-9a78-8bcff30dd0ac,1,Biotechnology & Nanotechnology Research,"[{'amount_raised_usd': '50M - 150M', 'valuatio...","[{'price_usd': 'Undisclosed', 'acquired_by_wel...","[\n {\n ""degree"": """",\n ""field"": """",\n ...","[\n {\n ""role"": ""Executive Chairman"",\n ...",This founder leads a startup in the Biotechnol...


In [3]:
# ── 2. Split BEFORE feature engineering  ───────────────────
train_data, test_data = train_test_split(
    data, test_size=0.20, random_state=42, stratify=data["success"]
)


print(f"Train → {(train_data.success==1).sum()} success / {(train_data.success==0).sum()} failure")
print(f"Test  → {(test_data.success==1).sum()} success / {(test_data.success==0).sum()} failure")

Train → 324 success / 3276 failure
Test  → 81 success / 819 failure


In [4]:
# ── 3. Feature engineering independently on each split ───────────────────────
df_train = build_feature_dataframe(train_data.to_dict("records"))
df_test  = build_feature_dataframe(test_data.to_dict("records"))

In [5]:
df_train.head()

,edu_count,has_md,has_phd,has_mba,has_postgrad,has_undergrad,best_qs_rank,has_top10_school,has_top50_school,has_top100_school,...,acq_any_large,acq_any_well_known_buyer,acq_undisclosed_count,industry_is_missing,industry_is_biotech,industry_is_software,industry_is_fintech,industry_is_health,success,founder_uuid
0,3,0,0,1,1,1,31,0,1,1,...,0,0,0,0,0,1,0,0,0,abe70c40-bc82-4353-b43e-f68769da4409
1,0,0,0,0,0,0,9999,0,0,0,...,0,0,0,1,0,0,0,0,1,f9747517-c066-496a-a476-4418bea9164a
2,0,0,0,0,0,0,9999,0,0,0,...,0,0,0,0,0,0,0,1,0,2adf1887-e866-4678-be07-74528f187436
3,1,0,0,0,0,1,200,0,0,0,...,0,0,0,0,0,0,0,0,0,39685292-6000-4166-b1ba-1b7dbf1456f9
4,1,0,0,0,0,1,200,0,0,0,...,0,0,0,0,0,1,0,0,0,568e7f96-a518-427e-aed9-1be9f269a89e


In [6]:
# ── 4. Separate features / target ────────────────────────────────────────────
X_train = df_train.drop(columns=["success", "founder_uuid"])
y_train = df_train["success"]

X_test  = df_test.drop(columns=["success", "founder_uuid"])
y_test  = df_test["success"]

print(f"\nX_train: {X_train.shape}  |  X_test: {X_test.shape}")


X_train: (3600, 40)  |  X_test: (900, 40)


In [7]:
from training_pipeline import train_all, load_data, compare_models

c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# ── 5. Train ──────────────────────────────────────────────────────────────────
pipeline = train_all(train_data.to_dict("records"), tune_hyperparams=False)


  XGBOOST
  3600 samples | class balance: 9.0% positive | pos_weight≈10.1
  Using default params.

  Metric        Test mean     ±std   Train mean
  --------------------------------------------
  roc_auc          0.6516   0.0236       0.9743
  avg_prec         0.1560   0.0200       0.8152
  f0.5             0.2194   0.0202       0.6017
  precision        0.1638   0.0177       0.4423
  recall           0.3334   0.0234       0.9414

  Saved → models/xgboost.pkl

  LIGHTGBM
  3600 samples | class balance: 9.0% positive | pos_weight≈10.1
  Using default params.

  Metric        Test mean     ±std   Train mean
  --------------------------------------------
  roc_auc          0.6497   0.0337       0.9640
  avg_prec         0.1655   0.0238       0.7545
  f0.5             0.2328   0.0355       0.5511
  precision        0.1663   0.0261       0.3922
  recall           0.3891   0.0577       0.9275

  Saved → models/lightgbm.pkl

  LOGREG
  3600 samples | class balance: 9.0% positive | pos_weight

In [10]:
# ── 6. Evaluate on held-out test set ─────────────────────────────────────────
X_test, y_test, _ = load_data(test_data.to_dict("records"))
comparison = compare_models(pipeline, X_test, y_test)



  MODEL COMPARISON — Held-out test set
               auc_roc  auc_pr    f0.5  precision  recall  threshold
model                                                               
logreg          0.7270  0.2612  0.3109     0.4286  0.1481      0.841
lightgbm        0.7137  0.2132  0.2846     0.2917  0.2593      0.692
xgboost         0.7115  0.1984  0.2878     0.2784  0.3333      0.620
random_forest   0.7128  0.1806  0.2456     0.2336  0.3086      0.513
knn             0.6323  0.1390  0.1784     0.1517  0.6049      0.200
  Winner (AUC-PR): logreg

📊  Comparison chart saved → model_comparison.png
